In [0]:

import pandas as pd
import numpy as np
     

In [0]:

caso_practico_ventas = spark.read.table("workspace.default.caso_practico_ventas").toPandas()

In [0]:
caso_practico_ventas

In [0]:
display(caso_practico_ventas.head(10))
display(caso_practico_ventas.tail(5))

In [0]:
# Conteo absoluto de nulos 
nulos_abs = caso_practico_ventas.isnull().sum()

In [0]:
# Porcentaje de nulos 
nulos_pct = (caso_practico_ventas.isnull().sum() / len(caso_practico_ventas)) * 100

In [0]:
# Consolidado en un DataFrame 
caso_practico_ventas_nulos = pd.DataFrame({'Total Nulos': nulos_abs, 'Porcentaje (%)': nulos_pct.round(2)}) 
print(caso_practico_ventas_nulos)

In [0]:
#Calculo de media, mediana, desviación estándar, mínimo y máximo para cantidad y precio_unitario.
columnas_num = ['cantidad', 'precio_unitario']
estadisticas = caso_practico_ventas[columnas_num].agg(['mean', 'median', 'std', 'min', 'max']).T
estadisticas.columns = ['Media', 'Mediana', 'Desviación Estándar', 'Mínimo', 'Máximo']
display(estadisticas)

In [0]:
# 1. Identificar cuántas filas duplicadas existen 
cant_duplicados = caso_practico_ventas.duplicated(subset=['id_transaccion']).sum() 
print(f"Total de registros duplicados: {cant_duplicados}")

In [0]:
# 2. Eliminar registros duplicados conservando el primero 
caso_practico_ventas_clean = caso_practico_ventas.drop_duplicates(subset=['id_transaccion'])

In [0]:
# 3. Verificar la nueva cantidad de filas 
print(f"Filas iniciales: {len(caso_practico_ventas)} | Filas tras eliminar duplicados: {len(caso_practico_ventas_clean)}")
     

In [0]:

# 1. Ver la frecuencia de valores antes de corregir 
print("Antes de estandarizar:")
print(caso_practico_ventas_clean['metodo_pago'].value_counts()) 

In [0]:
# 2. Reemplazar la variante inconsistente 
caso_practico_ventas_clean['metodo_pago'] = caso_practico_ventas_clean['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})

In [0]:
# 3. Ver la frecuencia corregida 
print("\nDespués de estandarizar:") 
print(caso_practico_ventas_clean['metodo_pago'].value_counts())

In [0]:

# 1. Calcular la mediana de 'cantidad' y la moda de 'satisfaccion\_cliente' 
mediana_cantidad = caso_practico_ventas_clean['cantidad'].median()
moda_satisfaccion = caso_practico_ventas_clean['satisfaccion_cliente'].mode()[0] 
print(f"Mediana calculada para cantidad: {mediana_cantidad}") 
print(f"Moda calculada para satisfacción: {moda_satisfaccion}") 

In [0]:
# 2. Imputar (rellenar) los valores nulos 
caso_practico_ventas_clean['cantidad'] = caso_practico_ventas_clean['cantidad'].fillna(mediana_cantidad)
caso_practico_ventas_clean['satisfaccion_cliente'] = caso_practico_ventas_clean['satisfaccion_cliente'].fillna(moda_satisfaccion)

In [0]:
# 3. Verificar que ya no existan nulos en estas dos columnas 
print("Nulos restantes en cantidad:", caso_practico_ventas_clean['cantidad'].isnull().sum()) 
print("Nulos restantes en satisfacción:", caso_practico_ventas_clean['satisfaccion_cliente'].isnull().sum())

In [0]:
# 1. Convertir el valor atípico de 9999.0 a NaN para no afectar el promedio 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean['precio_unitario'].replace(9999.0, np.nan)

In [0]:
# 2. Calcular la media de precio por categoría (sin incluir el outlier) e imputar 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean.groupby('categoria_producto')['precio_unitario'].transform( lambda x: x.fillna(x.mean()) ) 


In [0]:
# 3. Verificar que no queden nulos ni valores atípicos 
print("Nulos restantes en precio_unitario:", caso_practico_ventas_clean['precio_unitario'].isnull().sum()) 
print("Precio máximo corregido:", caso_practico_ventas_clean['precio_unitario'].max())

In [0]:
# 1. Columna derivada del importe total 
caso_practico_ventas_clean['monto_total'] =  caso_practico_ventas_clean['cantidad'] *  caso_practico_ventas_clean['precio_unitario'] 

In [0]:
# 2. Conversión temporal y extracción de día y mes 
caso_practico_ventas_clean['fecha'] = pd.to_datetime( caso_practico_ventas_clean['fecha'])
caso_practico_ventas_clean['dia_semana'] =  caso_practico_ventas_clean['fecha'].dt.day_name()
caso_practico_ventas_clean['mes'] =  caso_practico_ventas_clean['fecha'].dt.month_name()

In [0]:
# 3. Discretización en rangos de consumo 
def categorizar_monto(monto):
    if monto < 100:
        return 'Bajo'
    elif monto <= 500:
        return 'Medio'
    else:
        return 'Alto'

caso_practico_ventas_clean['rango_monto'] = caso_practico_ventas_clean['monto_total'].apply(categorizar_monto)

# Visualizar primeras filas con los nuevos atributos 
display(caso_practico_ventas_clean[['id_transaccion', 'monto_total', 'dia_semana', 'mes', 'rango_monto']].head())

In [0]:
# 1. Agrupamiento por Categoría 
resumen_cat = caso_practico_ventas_clean.groupby('categoria_producto').agg( total_ingresos=('monto_total', 'sum'), promedio_unidades=('cantidad', 'mean') ).round(2)

In [0]:
# 2. Tabla Dinámica (Región vs Categoría) 
tabla_dinamica = pd.pivot_table( caso_practico_ventas_clean, values='monto_total', index='region', columns='categoria_producto', aggfunc='sum', fill_value=0 ).round(2)

In [0]:
# 3. Satisfacción por Método de Pago 
satisfaccion_pago = caso_practico_ventas_clean.groupby('metodo_pago')['satisfaccion_cliente'].mean().round(2)
display(resumen_cat)
display(tabla_dinamica)
display(satisfaccion_pago)

In [0]:
# 1. Filtrado Multicriterio: Región Norte y Satisfacción >= 4 
filtro_norte_sat = caso_practico_ventas_clean[ (caso_practico_ventas_clean['region'] == 'Norte') & (caso_practico_ventas_clean['satisfaccion_cliente'] >= 4) ]

In [0]:
# 2. Muestreo Estratificado (10% por Región) 
muestra_estratificada = caso_practico_ventas_clean.groupby('region', group_keys=False).apply( lambda x: x.sample(frac=0.10, random_state=42) ) 

In [0]:
# Mostrar resultados 
print(f"Registros filtrados (Norte & Sat >= 4): {len(filtro_norte_sat)}")
display(filtro_norte_sat.head()) 
print(f"\Tamaño de la muestra estratificada: {len(muestra_estratificada)}")
display(muestra_estratificada)